# 9. Feature Engineering <a id="9-feature-engineering" name="9-feature-engineering"></a>

In [ ]:
# Text features for TF-IDF
def clean_text(t):
    if pd.isna(t): return ""
    return re.sub(r"[^a-zA-Z0-9 ]", " ", str(t).lower()).strip()

df["genres_clean"]   = df["genres"].apply(
    lambda x: " ".join([g.strip().replace(" ","") for g in str(x).split(",")]) if pd.notna(x) else "")
df["director_clean"] = df["director"].apply(
    lambda x: str(x).replace(" ","").lower() if pd.notna(x) else "")
df["cast_clean"]     = df["cast"].apply(
    lambda x: " ".join([a.strip().replace(" ","").lower() for a in str(x).split(",")[:5]])
              if pd.notna(x) else "")
df["overview_clean"] = df["overview"].apply(clean_text)
df["tagline_clean"]  = df["tagline"].apply(clean_text)

# Weighted soup — genres 4x, director 3x, cast 2x, overview 2x, tagline 1x
df["soup"] = (
    df["genres_clean"]   * 4 + " " +
    df["director_clean"] * 3 + " " +
    df["cast_clean"]     * 2 + " " +
    df["overview_clean"] * 2 + " " +
    df["tagline_clean"]
)

# Normalised numeric features
scaler = MinMaxScaler()
num_feats = ["vote_average","vote_count","runtime","popularity"]
df[["rating_norm","votes_norm","runtime_norm","pop_norm"]] = scaler.fit_transform(
    df[num_feats].fillna(0))
df["year_norm"] = ((df["release_year"] - df["release_year"].min()) /
                   (df["release_year"].max() - df["release_year"].min() + 1)).fillna(0)

print(" Feature Engineering Summary:")
print(f"  Text soup avg length:  {df['soup'].str.len().mean():.0f} chars")
print(f"  Genres weight:         4x in soup")
print(f"  Director weight:       3x in soup")
print(f"  Cast weight:           2x in soup")
print(f"  Overview weight:       2x in soup")
print(f"  Normalised numerics:   {num_feats}")
print()
print("Sample soup for 'Star Wars':")
sample = df[df["title"].str.lower() == "star wars"]
if len(sample):
    print(sample["soup"].values[0][:300])

 Feature Engineering Summary:
  Text soup avg length:  825 chars
  Genres weight:         4x in soup
  Director weight:       3x in soup
  Cast weight:           2x in soup
  Overview weight:       2x in soup
  Normalised numerics:   ['vote_average', 'vote_count', 'runtime', 'popularity']

Sample soup for 'Star Wars':
Adventure Action ScienceFictionAdventure Action ScienceFictionAdventure Action ScienceFictionAdventure Action ScienceFiction georgelucasgeorgelucasgeorgelucas markhamill harrisonford carriefisher petercushing alecguinnessmarkhamill harrisonford carriefisher petercushing alecguinness princess leia is


In [ ]:
# TF-IDF matrix
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
    sublinear_tf=True
)
tfidf_matrix = tfidf.fit_transform(df["soup"].fillna(""))
df = df.reset_index(drop=True)
title_to_idx = pd.Series(df.index, index=df["title"].str.lower())

print(f" TF-IDF Matrix built:")
print(f"   Shape:         {tfidf_matrix.shape}")
print(f"   Vocabulary:    {len(tfidf.vocabulary_):,} terms")
print(f"   Sparsity:      {1 - tfidf_matrix.nnz/(tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.4%}")
print(f"   Title index:   {len(title_to_idx):,} entries")

 TF-IDF Matrix built:
   Shape:         (5261, 20000)
   Vocabulary:    20,000 terms
   Sparsity:      99.8097%
   Title index:   5,261 entries
